# External validation with patient-grouped resampling

Revised to account for recurrent admissions:

- development OOF threshold selection uses `StratifiedGroupKFold` with `subject_reference`;
- model/calibration holdout splits are patient-grouped;
- external confidence intervals use patient-level cluster bootstrap resampling;
- the external cohort remains a single untouched validation cohort (no external CV).


In [ ]:
from __future__ import annotations

import os
import warnings
from typing import Dict, Iterable, List, Mapping, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    precision_recall_curve,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    roc_curve,
    auc,
    brier_score_loss,
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.linear_model import LogisticRegression

from tabpfn_extensions import TabPFNClassifier, interpretability

warnings.filterwarnings("ignore")


data_fs_static_external = pd.read_csv("YOUR_PATH")
data_fs_static_train = pd.read_csv("YOUR_PATH")

# Patient identifier used only for grouping/resampling; do not include it as a predictor.
GROUP_COL = "subject_reference"

scale = 'yes'
feature_space = ['feature_1', 'feature_2', ...]

if GROUP_COL in feature_space:
    raise ValueError(f"{GROUP_COL!r} must not be included in feature_space.")

X_external, y_external = do_train_test_split(data_fs_static_external, feature_space, scale)
X_train, y_train = do_train_test_split(data_fs_static_train, feature_space, scale)


def align_groups_to_model_rows(
    source_df: pd.DataFrame,
    X: pd.DataFrame,
    group_col: str = GROUP_COL,
) -> pd.Series:
    """Safely align patient identifiers to the rows returned by preprocessing."""
    if group_col not in source_df.columns:
        raise KeyError(f"{group_col!r} not found in source dataframe.")
    if source_df[group_col].isna().any():
        raise ValueError(f"{group_col} contains missing values.")

    if X.index.equals(source_df.index):
        return source_df.loc[X.index, group_col].copy()

    ambiguous_reset_index = (
        len(X) < len(source_df)
        and isinstance(X.index, pd.RangeIndex)
        and X.index.start == 0
        and X.index.step == 1
    )
    if (
        not ambiguous_reset_index
        and source_df.index.is_unique
        and X.index.isin(source_df.index).all()
    ):
        groups = source_df.loc[X.index, group_col].copy()
        groups.index = X.index
        return groups

    if len(source_df) == len(X):
        warnings.warn(
            f"Could not map {group_col} by index; assuming preprocessing preserved row order. "
            "For maximum safety, preserve the original dataframe index in do_train_test_split()."
        )
        return pd.Series(source_df[group_col].to_numpy(), index=X.index, name=group_col)

    raise ValueError(
        f"Could not safely align {group_col} to model rows. Preserve the original row index "
        "through do_train_test_split(), or return patient identifiers alongside X/y."
    )


groups_train = align_groups_to_model_rows(data_fs_static_train, X_train)
groups_external = align_groups_to_model_rows(data_fs_static_external, X_external)

print(
    f"Training cohort: {len(X_train)} admissions, {groups_train.nunique()} unique patients\n"
    f"External cohort: {len(X_external)} admissions, {groups_external.nunique()} unique patients"
)


In [ ]:
RANDOM_SEED = 42
N_SPLITS_THRESHOLD = 5
N_BOOTSTRAPS = 2000

FIG_DIR = "figures"
SHAP_DIR = "shap_values"
OUT_DIR = "outputs"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(SHAP_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "legend.fontsize": 10,
        "axes.grid": True,
        "grid.alpha": 0.25,
    }
)


HF_TOKEN = os.environ.get("HF_TOKEN", None)
if HF_TOKEN is None:
    print("Note: HF_TOKEN not found in environment. If TabPFN requires it, set it before running.")
else:
    os.environ["HF_TOKEN"] = HF_TOKEN

print("Ready.")

def as_numpy(y: Iterable) -> np.ndarray:
    y_arr = np.asarray(y)
    return y_arr.reshape(-1)

def safe_confusion(yt: np.ndarray, yp: np.ndarray) -> Tuple[int, int, int, int]:
    cm = confusion_matrix(yt, yp, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return int(tn), int(fp), int(fn), int(tp)

def ci_mean(vals: Iterable[float], alpha: float = 0.05) -> Tuple[float, float, float]:
    v = np.asarray(list(vals), dtype=float)
    lo = np.percentile(v, 100 * (alpha / 2))
    hi = np.percentile(v, 100 * (1 - alpha / 2))
    return float(v.mean()), float(lo), float(hi)

def stable_roc_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob))

def stable_auprc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    return float(auc(rec, prec))

def summarize_bootstrap(boot: Mapping[str, List[float]]) -> pd.DataFrame:
    rows = []
    for k in ["accuracy","precision","recall","specificity","sensitivity","npv","f1","mcc","auc","auprc"]:
        vals = [v for v in boot[k] if not np.isnan(v)]
        m, lo, hi = ci_mean(vals)
        rows.append([k, m, lo, hi])
    return pd.DataFrame(rows, columns=["metric", "mean", "ci_low", "ci_high"])

def align_features(
    X_target: pd.DataFrame,
    X_reference: pd.DataFrame,
    fill_strategy: str = "mean",  # "mean" or "zero"
) -> pd.DataFrame:
    X_aligned = X_target.copy()
    missing = [c for c in X_reference.columns if c not in X_aligned.columns]
    if missing:
        if fill_strategy == "mean":
            fill_vals = X_reference[missing].mean()
            for c in missing:
                X_aligned[c] = float(fill_vals[c])
        elif fill_strategy == "zero":
            for c in missing:
                X_aligned[c] = 0.0
        else:
            raise ValueError(f"Unknown fill_strategy='{fill_strategy}'")
    return X_aligned.reindex(columns=X_reference.columns)

def thresholds_from_predictions(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)

    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    if len(thresh) == 0:
        return {"F1-optimal": 0.5, "MCC-optimal": 0.5, "Youden": 0.5}

    f1_vals = 2 * (prec * rec) / (prec + rec + 1e-12)
    t_f1 = float(thresh[int(np.nanargmax(f1_vals[:-1]))])

    mcc_vals = [matthews_corrcoef(y_true, (y_prob >= t).astype(int)) for t in thresh]
    t_mcc = float(thresh[int(np.nanargmax(mcc_vals))])

    youden_vals = []
    for t in thresh:
        yp = (y_prob >= t).astype(int)
        tn, fp, fn, tp = safe_confusion(y_true, yp)
        sens = tp / (tp + fn + 1e-12)
        spec = tn / (tn + fp + 1e-12)
        youden_vals.append(sens + spec - 1)
    t_youden = float(thresh[int(np.nanargmax(youden_vals))])

    return {"F1-optimal": t_f1, "MCC-optimal": t_mcc, "Youden": t_youden}


def get_oof_probabilities_tabpfn(
    X: pd.DataFrame,
    y: np.ndarray,
    groups: np.ndarray,
    device: str = "auto",
    n_splits: int = N_SPLITS_THRESHOLD,
    seed: int = RANDOM_SEED,
) -> np.ndarray:
    """Patient-grouped OOF probabilities for TabPFN on the development cohort."""
    y = as_numpy(y)
    groups = as_numpy(groups)
    if not (len(X) == len(y) == len(groups)):
        raise ValueError("X, y, and groups must have identical lengths.")

    oof = np.full(shape=(len(X),), fill_value=np.nan, dtype=float)
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for tr_idx, va_idx in cv.split(X, y, groups=groups):
        overlap = set(groups[tr_idx]).intersection(set(groups[va_idx]))
        if overlap:
            raise RuntimeError(f"Patient leakage detected in OOF CV: {len(overlap)} overlapping patients.")

        m = TabPFNClassifier(device=device)
        m.fit(X.iloc[tr_idx].to_numpy(), y[tr_idx])
        oof[va_idx] = m.predict_proba(X.iloc[va_idx].to_numpy())[:, 1]

    if np.isnan(oof).any():
        raise RuntimeError("OOF probabilities contain NaNs; check grouped CV/data.")
    return oof


def _prepare_cluster_indices(groups: np.ndarray) -> Tuple[np.ndarray, List[np.ndarray]]:
    """Precompute admission-row indices for each unique patient."""
    groups = as_numpy(groups)
    if pd.isna(groups).any():
        raise ValueError("Patient identifiers contain missing values.")
    unique_groups = pd.unique(groups)
    rows_by_group = [np.flatnonzero(groups == g) for g in unique_groups]
    return np.asarray(unique_groups, dtype=object), rows_by_group


def cluster_bootstrap_indices(
    rng: np.random.Generator,
    rows_by_group: List[np.ndarray],
) -> np.ndarray:
    """Sample patients with replacement and retain all admissions for sampled patients."""
    n_groups = len(rows_by_group)
    sampled = rng.integers(0, n_groups, size=n_groups)
    return np.concatenate([rows_by_group[j] for j in sampled])


def grouped_model_calibration_split(
    X: pd.DataFrame,
    y: np.ndarray,
    groups: np.ndarray,
    cal_holdout_frac: float,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """Create a stratified patient-grouped model/calibration holdout split.

    For cal_holdout_frac=0.20 this uses five StratifiedGroupKFold partitions and
    chooses the fold closest to the requested size and overall outcome prevalence.
    """
    y = as_numpy(y)
    groups = as_numpy(groups)
    if not (len(X) == len(y) == len(groups)):
        raise ValueError("X, y, and groups must have identical lengths.")
    if not 0 < cal_holdout_frac < 1:
        raise ValueError("cal_holdout_frac must lie between 0 and 1.")

    n_splits = max(2, int(round(1.0 / cal_holdout_frac)))
    n_unique_groups = len(pd.unique(groups))
    if n_unique_groups < n_splits:
        raise ValueError(
            f"Need at least {n_splits} unique patients for grouped calibration splitting; "
            f"found {n_unique_groups}."
        )

    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    overall_prev = float(np.mean(y))
    candidates = []

    for model_idx, cal_idx in cv.split(X, y, groups=groups):
        overlap = set(groups[model_idx]).intersection(set(groups[cal_idx]))
        if overlap:
            raise RuntimeError("Patient leakage detected in model/calibration split.")
        if len(np.unique(y[model_idx])) < 2 or len(np.unique(y[cal_idx])) < 2:
            continue
        size_diff = abs(len(cal_idx) / len(y) - cal_holdout_frac)
        prev_diff = abs(float(np.mean(y[cal_idx])) - overall_prev)
        candidates.append((size_diff + prev_diff, model_idx, cal_idx))

    if not candidates:
        raise RuntimeError("Could not construct a grouped calibration split containing both outcome classes.")

    _, model_idx, cal_idx = min(candidates, key=lambda x: x[0])
    return model_idx, cal_idx


def bootstrap_metrics_fixed_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    groups: np.ndarray,
    threshold: float,
    n_boot: int = N_BOOTSTRAPS,
    seed: int = RANDOM_SEED,
) -> Dict[str, List[float]]:
    """Admission-level metrics with patient-cluster bootstrap confidence intervals."""
    rng = np.random.default_rng(seed)
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)
    groups = as_numpy(groups)
    if not (len(y_true) == len(y_prob) == len(groups)):
        raise ValueError("y_true, y_prob, and groups must have identical lengths.")

    keys = ["accuracy","precision","recall","specificity","sensitivity","npv","f1","mcc","auc","auprc","tn","fp","fn","tp"]
    M: Dict[str, List[float]] = {k: [] for k in keys}
    _, rows_by_group = _prepare_cluster_indices(groups)

    for _ in range(n_boot):
        idx = cluster_bootstrap_indices(rng, rows_by_group)
        yt = y_true[idx]
        pr = y_prob[idx]
        yp = (pr >= threshold).astype(int)

        tn, fp, fn, tp = safe_confusion(yt, yp)
        spec = tn / (tn + fp + 1e-12)
        sens = tp / (tp + fn + 1e-12)
        npv = tn / (tn + fn + 1e-12)

        M["accuracy"].append(accuracy_score(yt, yp))
        M["precision"].append(precision_score(yt, yp, zero_division=0))
        M["recall"].append(recall_score(yt, yp, zero_division=0))
        M["specificity"].append(spec)
        M["sensitivity"].append(sens)
        M["npv"].append(npv)
        M["f1"].append(f1_score(yt, yp, zero_division=0))
        M["mcc"].append(matthews_corrcoef(yt, yp))
        M["auc"].append(stable_roc_auc(yt, pr))
        M["auprc"].append(stable_auprc(yt, pr))
        M["tn"].append(tn); M["fp"].append(fp); M["fn"].append(fn); M["tp"].append(tp)

    return M


def _logit(p: np.ndarray) -> np.ndarray:
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    return np.log(p) - np.log(1 - p)

def calibration_in_the_large(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = as_numpy(y_true)
    z = _logit(as_numpy(y_prob)).reshape(-1, 1)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000)
    lr.fit(z, y_true)
    return float(lr.intercept_[0])

def calibration_slope(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = as_numpy(y_true)
    z = _logit(as_numpy(y_prob)).reshape(-1, 1)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000)
    lr.fit(z, y_true)
    return float(lr.coef_[0][0])

def bootstrap_calibration(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    groups: np.ndarray,
    n_boot: int = N_BOOTSTRAPS,
    seed: int = RANDOM_SEED,
) -> Dict[str, List[float]]:
    """Calibration metrics with patient-cluster bootstrap confidence intervals."""
    rng = np.random.default_rng(seed)
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)
    groups = as_numpy(groups)
    if not (len(y_true) == len(y_prob) == len(groups)):
        raise ValueError("y_true, y_prob, and groups must have identical lengths.")

    brier_vals, citl_vals, slope_vals = [], [], []
    _, rows_by_group = _prepare_cluster_indices(groups)

    for _ in range(n_boot):
        idx = cluster_bootstrap_indices(rng, rows_by_group)
        yt = y_true[idx]
        pr = y_prob[idx]
        brier_vals.append(float(brier_score_loss(yt, pr)))

        # Logistic calibration metrics require both outcome classes.
        if len(np.unique(yt)) < 2:
            citl_vals.append(float("nan"))
            slope_vals.append(float("nan"))
        else:
            try:
                citl_vals.append(calibration_in_the_large(yt, pr))
                slope_vals.append(calibration_slope(yt, pr))
            except Exception:
                citl_vals.append(float("nan"))
                slope_vals.append(float("nan"))

    return {"brier": brier_vals, "citl": citl_vals, "slope": slope_vals}


def plot_and_save_roc(y_true: np.ndarray, y_prob: np.ndarray, name: str) -> None:
    y_true = as_numpy(y_true); y_prob = as_numpy(y_prob)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = stable_roc_auc(y_true, y_prob)

    plt.figure(figsize=(6.3, 4.6))
    plt.plot(fpr, tpr, label=f"AUC={auc_val:.3f}")
    plt.plot([0,1],[0,1],"--", linewidth=1)
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("ROC curve (TabPFN)")
    plt.legend(loc="lower right")
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_roc.png")
    plt.savefig(path)
    plt.show()
    plt.close()
    print(f"Saved: {path}")

def plot_and_save_pr(y_true: np.ndarray, y_prob: np.ndarray, name: str) -> None:
    y_true = as_numpy(y_true); y_prob = as_numpy(y_prob)
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    auprc_val = stable_auprc(y_true, y_prob)

    plt.figure(figsize=(6.3, 4.6))
    plt.plot(rec, prec, label=f"AUPRC={auprc_val:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision–Recall curve (TabPFN)")
    plt.legend(loc="lower left")
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_pr.png")
    plt.savefig(path)
    plt.show()
    plt.close()
    print(f"Saved: {path}")

def plot_and_save_calibration(y_true: np.ndarray, y_prob: np.ndarray, name: str, n_bins: int = 10) -> None:
    y_true = as_numpy(y_true); y_prob = as_numpy(y_prob)
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy="uniform")

    plt.figure(figsize=(6.3, 4.6))
    plt.plot(prob_pred, prob_true, marker="o", linewidth=1, label="Model")
    plt.plot([0,1],[0,1],"--", linewidth=1, label="Perfect")
    plt.xlabel("Predicted probability")
    plt.ylabel("Observed frequency")
    plt.title("Calibration curve (TabPFN)")
    plt.legend()
    plt.tight_layout()
    path = os.path.join(FIG_DIR, f"{name}_calibration.png")
    plt.savefig(path)
    plt.show()
    plt.close()
    print(f"Saved: {path}")

def get_oof_probabilities_tabpfn_with_optional_calibration(
    X: pd.DataFrame,
    y: np.ndarray,
    groups: np.ndarray,
    device: str = "auto",
    calibration_mode: str = "uncal",  # "uncal" | "platt" | "iso"
    cal_holdout_frac: float = 0.2,
    n_splits: int = N_SPLITS_THRESHOLD,
    seed: int = RANDOM_SEED,
) -> np.ndarray:
    """Patient-grouped OOF probabilities for leakage-free threshold selection.

    If calibration is requested, each OOF training partition is itself split into
    patient-disjoint model and calibration subsets.
    """
    y = as_numpy(y)
    groups = as_numpy(groups)
    if not (len(X) == len(y) == len(groups)):
        raise ValueError("X, y, and groups must have identical lengths.")

    oof = np.full(shape=(len(X),), fill_value=np.nan, dtype=float)
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y, groups=groups), start=1):
        X_tr = X.iloc[tr_idx]
        y_tr = y[tr_idx]
        groups_tr = groups[tr_idx]
        X_va = X.iloc[va_idx]

        overlap = set(groups_tr).intersection(set(groups[va_idx]))
        if overlap:
            raise RuntimeError(
                f"Patient leakage detected in threshold OOF fold {fold}: "
                f"{len(overlap)} overlapping patients."
            )

        if calibration_mode == "uncal":
            m = TabPFNClassifier(device=device)
            m.fit(X_tr.to_numpy(), y_tr)
            oof[va_idx] = m.predict_proba(X_va.to_numpy())[:, 1]

        elif calibration_mode in ("platt", "iso"):
            model_rel_idx, cal_rel_idx = grouped_model_calibration_split(
                X=X_tr,
                y=y_tr,
                groups=groups_tr,
                cal_holdout_frac=cal_holdout_frac,
                seed=seed + fold,
            )
            X_model = X_tr.iloc[model_rel_idx]
            y_model = y_tr[model_rel_idx]
            X_cal = X_tr.iloc[cal_rel_idx]
            y_cal = y_tr[cal_rel_idx]

            base = TabPFNClassifier(device=device)
            base.fit(X_model.to_numpy(), y_model)

            method = "sigmoid" if calibration_mode == "platt" else "isotonic"
            cal = CalibratedClassifierCV(base, method=method, cv="prefit")
            cal.fit(X_cal.to_numpy(), y_cal)
            oof[va_idx] = cal.predict_proba(X_va.to_numpy())[:, 1]
        else:
            raise ValueError("calibration_mode must be 'uncal', 'platt', or 'iso'")

    if np.isnan(oof).any():
        raise RuntimeError("OOF probabilities contain NaNs; check grouped CV/data.")
    return oof


def run_pipeline_tabpfn_external(
    X_train: pd.DataFrame,
    y_train: Iterable,
    X_external: pd.DataFrame,
    y_external: Iterable,
    groups_train: Iterable,
    groups_external: Iterable,
    feature_fill_strategy: str = "mean",
    device: str = "auto",
    calibration_mode: str = "uncal",  # "uncal" | "platt" | "iso"
    cal_holdout_frac: float = 0.2,
    n_splits_threshold: int = N_SPLITS_THRESHOLD,
    n_bootstraps: int = N_BOOTSTRAPS,
    run_shap: bool = True,
    max_shap_samples: int = 1000,
    prefix: str = "tabpfn_external",
) -> Dict[str, Dict[str, List[float]]]:
    y_tr = as_numpy(y_train)
    y_ext = as_numpy(y_external)
    g_tr = as_numpy(groups_train)
    g_ext = as_numpy(groups_external)

    if not (len(X_train) == len(y_tr) == len(g_tr)):
        raise ValueError("X_train, y_train, and groups_train must have identical lengths.")
    if not (len(X_external) == len(y_ext) == len(g_ext)):
        raise ValueError("X_external, y_external, and groups_external must have identical lengths.")
    if pd.isna(g_tr).any() or pd.isna(g_ext).any():
        raise ValueError("Patient identifiers must not contain missing values.")

    print(
        f"Development cohort: {len(y_tr)} admissions / {len(pd.unique(g_tr))} patients\n"
        f"External cohort: {len(y_ext)} admissions / {len(pd.unique(g_ext))} patients"
    )

    # --- 1) Threshold selection from TRAINING OOF probabilities (patient-grouped; no leakage)
    print(f"Computing TRAINING OOF probabilities for threshold selection (mode={calibration_mode})...")
    p_oof = get_oof_probabilities_tabpfn_with_optional_calibration(
        X=X_train,
        y=y_tr,
        groups=g_tr,
        device=device,
        calibration_mode=calibration_mode,
        cal_holdout_frac=cal_holdout_frac,
        n_splits=n_splits_threshold,
        seed=RANDOM_SEED,
    )
    thresholds = thresholds_from_predictions(y_tr, p_oof)
    thr_df = pd.DataFrame({"rule": list(thresholds.keys()), "threshold": list(thresholds.values())})
    thr_path = os.path.join(OUT_DIR, f"{prefix}_thresholds_from_train_oof_{calibration_mode}.csv")
    thr_df.to_csv(thr_path, index=False)
    print("\nFrozen thresholds (TRAINING OOF):")
    display(thr_df)
    print(f"Saved: {thr_path}")

    # --- 2) Fit final model on training (and calibrate if requested)
    X_ext = align_features(X_external, X_train, fill_strategy=feature_fill_strategy)

    if calibration_mode == "uncal":
        final_model = TabPFNClassifier(device=device)
        final_model.fit(X_train.to_numpy(), y_tr)
        p_ext = final_model.predict_proba(X_ext.to_numpy())[:, 1]
        base_for_shap = final_model
        X_shap_train = X_train

    elif calibration_mode in ("platt", "iso"):
        model_idx, cal_idx = grouped_model_calibration_split(
            X=X_train,
            y=y_tr,
            groups=g_tr,
            cal_holdout_frac=cal_holdout_frac,
            seed=RANDOM_SEED,
        )
        X_model = X_train.iloc[model_idx]
        y_model = y_tr[model_idx]
        X_cal = X_train.iloc[cal_idx]
        y_cal = y_tr[cal_idx]

        model_patients = set(g_tr[model_idx])
        cal_patients = set(g_tr[cal_idx])
        if model_patients.intersection(cal_patients):
            raise RuntimeError("Patient leakage detected in final model/calibration split.")
        print(
            f"Final grouped calibration split: "
            f"{len(model_idx)} model admissions / {len(model_patients)} patients; "
            f"{len(cal_idx)} calibration admissions / {len(cal_patients)} patients."
        )

        base = TabPFNClassifier(device=device)
        base.fit(X_model.to_numpy(), y_model)

        method = "sigmoid" if calibration_mode == "platt" else "isotonic"
        final_cal = CalibratedClassifierCV(base, method=method, cv="prefit")
        final_cal.fit(X_cal.to_numpy(), y_cal)

        p_ext = final_cal.predict_proba(X_ext.to_numpy())[:, 1]
        base_for_shap = base
        X_shap_train = X_model

    else:
        raise ValueError("calibration_mode must be 'uncal', 'platt', or 'iso'")

    # --- 3) External plots
    print("\nExternal ROC/PR/Calibration plots:")
    plot_and_save_roc(y_ext, p_ext, name=f"{prefix}_{calibration_mode}")
    plot_and_save_pr(y_ext, p_ext, name=f"{prefix}_{calibration_mode}")
    plot_and_save_calibration(y_ext, p_ext, name=f"{prefix}_{calibration_mode}")

    # --- 4) External performance at fixed thresholds + patient-cluster bootstrap
    results: Dict[str, Dict[str, List[float]]] = {}
    summaries = []

    for rule, thr in thresholds.items():
        boot = bootstrap_metrics_fixed_threshold(
            y_ext, p_ext, groups=g_ext, threshold=thr, n_boot=n_bootstraps
        )
        results[rule] = boot

        df = summarize_bootstrap(boot)
        df.insert(0, "rule", rule)
        df.insert(1, "threshold", thr)
        df.insert(2, "calibration_mode", calibration_mode)
        summaries.append(df)

    perf_df = pd.concat(summaries, ignore_index=True)
    perf_path = os.path.join(OUT_DIR, f"{prefix}_external_bootstrap_metrics_{calibration_mode}.csv")
    perf_df.to_csv(perf_path, index=False)

    print("\nExternal performance summary (bootstrapped mean + 95% CI):")
    display(perf_df)
    print(f"Saved: {perf_path}")

    # --- 5) External calibration analysis + patient-cluster bootstrap CIs
    print("\nCalibration analysis (External):")
    brier = float(brier_score_loss(y_ext, p_ext))
    citl = calibration_in_the_large(y_ext, p_ext)
    slope = calibration_slope(y_ext, p_ext)
    print(f"Brier score (point):       {brier:.4f}")
    print(f"CITL (point):              {citl:.4f}")
    print(f"Calibration slope (point): {slope:.4f}")

    C = bootstrap_calibration(y_ext, p_ext, groups=g_ext, n_boot=n_bootstraps)
    cal_rows = []
    for label, key in [("Brier", "brier"), ("CITL", "citl"), ("Slope", "slope")]:
        vals = [v for v in C[key] if not np.isnan(v)]
        m, lo, hi = ci_mean(vals)
        cal_rows.append([label, m, lo, hi, calibration_mode])

    cal_df = pd.DataFrame(cal_rows, columns=["metric", "mean", "ci_low", "ci_high", "calibration_mode"])
    cal_path = os.path.join(OUT_DIR, f"{prefix}_external_calibration_{calibration_mode}.csv")
    cal_df.to_csv(cal_path, index=False)
    display(cal_df)
    print(f"Saved: {cal_path}")

    # --- 6) TabPFN interpretability (permutation SHAP) on subsamples
    if run_shap:
        print("\nComputing TabPFN SHAP-like values (permutation; subsampled)...")
        run_tabpfn_permutation_shap(
            estimator=base_for_shap,
            X_train_for_shap=X_shap_train,
            X_external_aligned=X_ext,
            feature_names=list(X_train.columns.astype(str)),
            prefix=f"tabpfn_{calibration_mode}",
            max_samples=max_shap_samples,
        )

    print("\nDone.")
    return results

def run_tabpfn_permutation_shap(
    estimator: TabPFNClassifier,
    X_train_for_shap: pd.DataFrame,
    X_external_aligned: pd.DataFrame,
    feature_names: List[str],
    prefix: str = "tabpfn",
    max_samples: int = 1000,
    seed: int = RANDOM_SEED,
) -> None:
    def subsample_np(X: np.ndarray, n: int) -> np.ndarray:
        if len(X) <= n:
            return X
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(X), n, replace=False)
        return X[idx]

    X_train_np = X_train_for_shap.to_numpy(dtype=np.float64)
    X_ext_np = X_external_aligned.to_numpy(dtype=np.float64)

    X_train_sub = subsample_np(X_train_np, max_samples)
    X_ext_sub = subsample_np(X_ext_np, max_samples)

    shap_train = interpretability.shap.get_shap_values(
        estimator=estimator,
        test_x=X_train_sub,
        attribute_names=feature_names,
        algorithm="permutation",
    )
    shap_ext = interpretability.shap.get_shap_values(
        estimator=estimator,
        test_x=X_ext_sub,
        attribute_names=feature_names,
        algorithm="permutation",
    )

    # The API can return different shapes depending on version.
    # We standardize to a numpy array (n_samples, n_features) when possible.
    shap_train_arr = np.asarray(shap_train)
    shap_ext_arr = np.asarray(shap_ext)

    def to_matrix(arr: np.ndarray) -> np.ndarray:
        # If binary class dimension exists, pick positive class.
        # Common possibilities: (2, n, p) or (n, p) or (n, p, 2)
        if arr.ndim == 3 and arr.shape[0] == 2:
            return arr[1]
        if arr.ndim == 3 and arr.shape[-1] == 2:
            return arr[..., 1]
        if arr.ndim == 2:
            return arr
        raise ValueError(f"Unexpected SHAP array shape: {arr.shape}")

    S_tr = to_matrix(shap_train_arr)
    S_ext = to_matrix(shap_ext_arr)

    def save_mean_abs(S: np.ndarray, name: str) -> str:
        mean_abs = np.abs(S).mean(axis=0)
        df = pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs}).sort_values(
            "mean_abs_shap", ascending=False
        )
        out = os.path.join(SHAP_DIR, f"{prefix}_shap_{name}.csv")
        df.to_csv(out, index=False)
        return out

    p1 = save_mean_abs(S_tr, "train")
    p2 = save_mean_abs(S_ext, "external")
    print(f"Saved: {p1}")
    print(f"Saved: {p2}")

# Requires: X_train, y_train, X_external, y_external

results_tabpfn = run_pipeline_tabpfn_external(
    X_train=X_train,
    y_train=y_train,
    X_external=X_external,
    y_external=y_external,
    groups_train=groups_train,
    groups_external=groups_external,
    feature_fill_strategy="mean",
    device="auto",
    calibration_mode="platt",   # "uncal" / "platt" / "iso"
    cal_holdout_frac=0.2,
    n_splits_threshold=N_SPLITS_THRESHOLD,
    n_bootstraps=N_BOOTSTRAPS,
    run_shap=True,
    max_shap_samples=1000,
    prefix="tabpfn_external",
)
